In [1]:
%pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.1 MB/s eta 0:00:0031m12.2 MB/s eta 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993225 sha256=14e1e3cf187d8ebf456070ef878c4354c27c264dcd6b9d13b357e2477ff04a52
  Stored in directory: /Users/yumiko/Library/Caches/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [2]:
## load raw data
import pandas as pd  
import glob          # for finding files by pattern

# find all four CSV files in data/raw and combine them into one table
files = glob.glob("../data/raw/*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Files loaded:", len(files))  # should say 4
print("Total reviews:", len(df))     # should be about 58,500

Files loaded: 4
Total reviews: 58493


In [ ]:
# fix date column
# CSV files store dates as text; convert them back into real dates
df["at"] = pd.to_datetime(df["at"])

print(df["at"].min(), "to", df["at"].max())  # check the date range

In [3]:
# remove duplicates and empty reviews
before = len(df)  # remember how many reviews we started with

df = df.drop_duplicates(subset="reviewId")  # every review has a unique ID; remove repeats
df = df.dropna(subset=["content"])          # remove reviews with no text at all
df = df[df["content"].str.strip() != ""]    # remove reviews that are only spaces

print("Removed:", before - len(df), "reviews")
print("Remaining:", len(df))

Removed: 7 reviews
Remaining: 58486


In [4]:
# flag gibberish and non-English reviews
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # makes language detection give the same result every run

def detect_language(text):
    try:
        return detect(text)   # returns a code like "en" for English
    except:
        return "unknown"      # happens with emoji-only or empty text

df["word_count"] = df["content"].str.split().str.len()   # number of words in each review
df["language"] = df["content"].apply(detect_language)    # detect each review's language

# text is usable if it's English and has at least 3 words
df["usable_text"] = (df["language"] == "en") & (df["word_count"] >= 3)

print(df["usable_text"].value_counts())  # how many usable vs not usable
# Why "at least 3 words"? 
# Language detection is unreliable on very short text, 
#and one- or two-word reviews like "great" don't give a classifier much to learn from anyway.

usable_text
True     48499
False     9987
Name: count, dtype: int64


In [6]:
## add useful columns

df["at"] = pd.to_datetime(df["at"], errors="coerce")  # make sure "at" is a real date
print("Date type:", df["at"].dtype, "| Unreadable dates:", df["at"].isna().sum())  # check it worked

# month of each review, e.g. "2024-03", for trends over time
df["month"] = df["at"].dt.to_period("M").astype(str)

# which period of the Mint shutdown each review falls in
# Mint's shutdown was announced Nov 2, 2023 and it closed Mar 23, 2024
def mint_period(date):
    if date < pd.Timestamp("2023-11-02"):
        return "1_before_announcement"
    elif date < pd.Timestamp("2024-03-23"):
        return "2_transition"
    else:
        return "3_after_shutdown"

df["mint_period"] = df["at"].apply(mint_period)

# True/False: does the review mention Mint?
df["mentions_mint"] = df["content"].str.contains(r"\bmint\b", case=False, na=False)

print(df["mint_period"].value_counts().sort_index())  # reviews per period

Date type: datetime64[ns] | Unreadable dates: 0
mint_period
1_before_announcement     7826
2_transition              9851
3_after_shutdown         40809
Name: count, dtype: int64


In [7]:
## save cleaned data
# keep only the columns we need, in a clear order
columns = ["reviewId", "app_name", "score", "content", "at", "month", "mint_period",
           "reviewCreatedVersion", "thumbsUpCount", "word_count", "language",
           "usable_text", "mentions_mint"]
df = df[columns]

df.to_csv("../data/processed/reviews_clean.csv", index=False)  # save as CSV
print("Saved", len(df), "reviews")

Saved 58486 reviews


In [ ]:
## save to SQL

import sqlite3  # built-in database library

conn = sqlite3.connect("../data/processed/reviews.db")  # creates the database file

# write the table into the database (replace it if it already exists)
df.to_sql("reviews", conn, if_exists="replace", index=False)

# test it with a first SQL query: reviews and average rating per app
test = pd.read_sql("""
    SELECT app_name, COUNT(*) AS reviews, ROUND(AVG(score), 2) AS avg_rating
    FROM reviews
    GROUP BY app_name
""", conn)

conn.close()  # close the connection when done
test